### Automatische Anpassung der Statistics.md des Git-Repositories

In [1]:
import pandas as pd
import datetime
import os

# Pfade definieren
ASSETS_DIR = "../assets/"
DATA_DIR = "../data/"
OUTPUT_FILE = "../Statistics.md"

# 1. Lade die Evaluationstabelle & Performance Summary (als String)
table_path = os.path.join(ASSETS_DIR, "evaluation_table.md")
with open(table_path, "r", encoding="utf-8") as f:
    evaluation_table_md = f.read()
summary_path = os.path.join(ASSETS_DIR, "performance_summary.md")
with open(summary_path, "r", encoding="utf-8") as f:
    performance_summary_md = f.read()
with open(os.path.join(ASSETS_DIR, "sorr_summary.md"), "r", encoding="utf-8") as f:
    sorr_summary_md = f.read()
with open(os.path.join(ASSETS_DIR, "mcs_summary.md"), "r", encoding="utf-8") as f:
    mcs_summary_md = f.read()

# 2. Erstelle den dynamischen Inhalt
timestamp = datetime.datetime.now().strftime("%d.%m.%Y %H:%M")

stats_md_content = f"""
# Detaillierte statistische Auswertung & Forschungsergebnisse

Diese Seite dokumentiert die numerischen und grafischen Ergebnisse der Forschungs-Pipeline. Alle Auswertungen basieren auf dem Datensatz bis zum gestrigen Tag und werden automatisiert aktualisiert.

---

## 1. Executive Summary: Performance & Risiko
Ein direkter Vergleich der Kernkennzahlen über den gesamten **Out-of-Sample Testzeitraum**.

{performance_summary_md}

> **Kernaussage:** Vergleiche den **Max Drawdown** der aktiven Strategien mit der Buy & Hold Benchmark. Ziel der Arbeit ist eine signifikante Reduktion dieses Werts zur Minderung des SORR.

---

## 2. Datenbasis & Baseline Portfolio
Grundlage der Untersuchung ist ein globaler Multi-Asset-Ansatz.

### 60/40 Portfolio Kapitalkurve
Die Abbildung zeigt die kumulierte Wertentwicklung des statischen Referenzportfolios (60% Aktien / 40% Anleihen).

![Capital Curve](./assets/capital_curve.png)

*   **Datenquelle:** S&P 500 (`^GSPC`) und Vanguard Long-Term Treasury (`VUSTX`).
*   **Reproduzierbarkeit:** Der bereinigte Datensatz inkl. aller Features ist hinterlegt unter: `data/02_feature_engineered_data.parquet`.

---

## 3. Regime-Erkennung der Einzelmodelle
Hier werden die Identifikations-Ergebnisse der drei Modell-Kategorien (Statistik, Clustering, Deep Learning) visualisiert.

### A. Hidden Markov Model (Unsupervised Clustering)
![HMM Regimes](./assets/hmm_regimes.png)

### B. Markov-Switching-Modelle (Ökonometrie)
Vergleich zwischen univariatem Ansatz und exogenem Ansatz (unter Berücksichtigung von VIX & Yield Spread).
![Markov Models](./assets/markov-models.png)

### C. LSTM-Netzwerk (Deep Learning)
Vorhersage der Marktphasen durch das neuronale Netzwerk (trainiert auf Markov-Labels).
![LSTM Model](./assets/lstm_model.png)

### D. Unsupervised LSTM-Netzwerk (Deep Learning)
Identifikation von Marktregimes mittels eines LSTM-Autoencoders in Kombination mit Gaussian Mixture Modeling (GMM). Im Gegensatz zum Supervised-Ansatz lernt dieses Modell ohne vordefinierte Labels (wie HMM oder Markov) und identifiziert Regime-Strukturen rein datengetrieben durch die Kompression und Rekonstruktion zeitlicher Sequenzen.
![Unsupervised LSTM Model](./assets/lstm_unsupervised_model.png)

### E. Globaler Regime-Vergleich
Detaillierte Gegenüberstellung der Wahrscheinlichkeiten und harten Signale aller Modelle.
![Regime Comparison](./assets/regime_comparison.png)

---

## 4. Backtesting & Strategie-Evaluation
Die ökonomische Anwendung der Regime-Signale durch dynamische Umschichtung in den Geldmarkt.

### Equity Curves im Vergleich
![Equity Curves](./assets/equity_curves.png)

### Umfassende Kennzahlen-Matrix
Detaillierte statistische Analyse inklusive risikoadjustierter Kennzahlen (Sharpe, Sortino, Calmar).

{evaluation_table_md}

### Transaktionskosten

Diese Grafik zeigt die kumulierten Transaktionskosten im Zeitverlauf. Steile Anstiege deuten auf instabile Regime-Wechsel ("Churning") hin.

![Transaction Costs](./assets/transaction_costs.png)

Stress-Test: Sequence of Returns Risk (SORR)
Außerdem wurde die Überlebensdauer des Kapitals in einer simulierten Entnahmephase (Ruhestandsszenario) durchgeführt.

### SORR-Simulation: Vergleich der Entnahmeszenarien

In dieser Tabelle werden verschiedene Stress-Szenarien (Standard, Aggressiv, Geringes Kapital) gegenübergestellt.

{sorr_summary_md}

Abbildung der Kapitalentwicklung der unterschiedlichen Szenarien:
![SORR Standard](./assets/sorr_sim_standard.png)
![SORR Aggressive](./assets/sorr_sim_aggressive.png)
![SORR Low Capital](./assets/sorr_sim_low_capital.png)

### MCS: Block-Bootstrap Robustness-Check

Um die statistische Signifikanz zu prüfen, wurden 1.000 künstliche Marktpfade mittels Block-Bootstrap simuliert.
![MCS Paths](./assets/mcs_paths.png)
{mcs_summary_md}

Verteilung der Endkapitalwerte
![MCS Boxplots Standard](./assets/mcs_boxplot_standard.png)
![MCS Boxplots Aggressive](./assets/mcs_boxplot_aggressive.png)
![MCS Boxplots Low Capital](./assets/mcs_boxplot_low_capital.png)

Wahrscheinlichkeitskorridore (Fan-Charts)
Die schattierten Bereiche zeigen das 5% bis 95% Konfidenzintervall der Kapitalentwicklung.
![MCS Quantiles](./assets/mcs_quantiles.png)

---

## 📝 Forschungsnotizen & Methodik
- **Cash-Komponente:** Bei einem "Bear"-Signal schichtet die Strategie in den aktuellen Geldmarktzins (**^IRX**) um.
- **Vermeidung von Look-ahead Bias:** Alle Signale werden für das Backtesting um einen Tag zeitversetzt (`shift(1)`), um reale Handelsbedingungen zu simulieren.
- **Feature-Set:** Die Modelle nutzen Renditen, Volatilität (20d), SMA-Abstand, Momentum, VIX und Yield Spread.
- **Kostensimulation:** Es wird eine pauschale Gebühr von 10 Basispunkten (0,1%) pro Umschichtung berechnet.
- **SORR-Spezifika:** Bei Entnahmen in "Bull"-Phasen wird eine zusätzliche Liquiditätsgebühr von 0,1% auf den Entnahmebetrag erhoben (Asset-Verkäufe). In "Bear"-Phasen (Cash) entfällt diese.

---
**Zuletzt aktualisiert:** {timestamp}  
*Generiert durch die automatisierte ETL-Pipeline (Notebook 99).*
"""

# 3. Datei schreiben
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(stats_md_content)

print(f"Statistics.md wurde erfolgreich aktualisiert ({timestamp}).")

Statistics.md wurde erfolgreich aktualisiert (03.02.2026 15:27).
